# Full Fine-tuning — Llama-3.2-1B-Instruct

**Strategy:** Full Fine-tuning — cập nhật tất cả trọng số của toàn bộ model.

**Model:** `meta-llama/Llama-3.2-1B-Instruct` (1B params)

**Dataset:** `code_x_glue_cc_defect_detection` (~27k samples, C/C++ defect detection)

**Ưu điểm:** Hiệu quả cao nhất vì toàn bộ model được tối ưu cho task cụ thể.

**Nhược điểm:** Tốn tài nguyên, dễ overfit nếu data nhỏ.

Model 1B đủ nhỏ để full fine-tune trên Colab T4 16GB.

## 1. Setup

In [ ]:
!pip install -q transformers datasets accelerate bitsandbytes
!pip install -q trl peft scikit-learn matplotlib seaborn huggingface_hub

from huggingface_hub import login
login()  # Llama 3.2 is a gated model — paste your HF token when prompted

In [ ]:
import os
import sys
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from trl import SFTTrainer

# On Colab, clone the repo first:
# !git clone -b trung https://github.com/MinhTuan2405/IE105.21_SBugDwLLM.git
# %cd IE105.21_SBugDwLLM/finetuning/llama32

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from utils.data_loader import load_defect_dataset, format_for_finetuning, SYSTEM_PROMPT
from utils.evaluation import evaluate_predictions, parse_model_output

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
OUTPUT_DIR = "./llama32-full-finetuned"
RESULTS_DIR = "../../docs/llama32_docs"

MAX_SEQ_LENGTH = 1024
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

## 3. Load Dataset

In [ ]:
train_data, val_data, test_data = load_defect_dataset()

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

train_labels = [s["target"] for s in train_data]
print(f"Train distribution: 0={train_labels.count(0)}, 1={train_labels.count(1)}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

train_formatted = train_data.map(lambda x: format_for_finetuning(x, tokenizer=tokenizer))
val_formatted = val_data.map(lambda x: format_for_finetuning(x, tokenizer=tokenizer))

print("\nSample formatted:")
print(train_formatted[0]["text"][:500])

## 4. Load Model (Full Precision for Full Fine-tuning)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Full fine-tuning: all parameters are trainable
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
print(f"Memory: {model.get_memory_footprint() / 1e9:.2f} GB")

## 5. Training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    dataset_text_field="text",
)

print("Starting FULL fine-tuning...")
trainer.train()

In [ ]:
trainer.save_model(os.path.join(OUTPUT_DIR, "best"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "best"))
print(f"Model saved to {OUTPUT_DIR}/best")

## 6. Evaluation

In [ ]:
from tqdm import tqdm
from utils.data_loader import build_chat_messages

model.eval()
device = next(model.parameters()).device
y_true, y_pred, failed_parses = [], [], []

for i, sample in enumerate(tqdm(test_data, desc="Evaluating")):
    messages = build_chat_messages(sample["func"])
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=5, do_sample=False)

    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    pred = parse_model_output(generated)

    if pred == -1:
        failed_parses.append({"index": i, "output": generated, "label": sample["target"]})
        pred = 0

    y_true.append(sample["target"])
    y_pred.append(pred)

print(f"\nFailed to parse: {len(failed_parses)} / {len(test_data)}")

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

metrics = evaluate_predictions(
    y_true, y_pred,
    model_name="llama32",
    strategy="full_finetuning",
    save_dir=RESULTS_DIR,
)

## 7. Error Analysis

In [ ]:
errors = [
    {"index": i, "true": yt, "pred": yp, "code": test_data[i]["func"][:200]}
    for i, (yt, yp) in enumerate(zip(y_true, y_pred))
    if yt != yp
]

false_positives = [e for e in errors if e["true"] == 0 and e["pred"] == 1]
false_negatives = [e for e in errors if e["true"] == 1 and e["pred"] == 0]

print(f"Total errors: {len(errors)}")
print(f"False Positives (safe -> defective): {len(false_positives)}")
print(f"False Negatives (defective -> safe): {len(false_negatives)}")

print("\n--- Sample False Negatives ---")
for e in false_negatives[:3]:
    print(f"  idx={e['index']}: {e['code'][:100]}...")

print("\n--- Sample False Positives ---")
for e in false_positives[:3]:
    print(f"  idx={e['index']}: {e['code'][:100]}...")